# Finviz Stock Metrics Scraper

Extract valuation metrics from Finviz for a configurable ticker. The first test ticker is AMD. This notebook uses Finviz only; it does not switch data providers.

**Fields:** Ticker, P/E, Forward P/E, P/S, P/C, P/B and P/FCF.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from playwright.async_api import async_playwright
from IPython.display import display

TICKER = os.getenv("TICKER", "AMD").strip().upper()  # Set by GitHub Actions or change the default
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ["P/E", "Forward P/E", "P/S", "P/C", "P/B", "P/FCF"]
FINVIZ_URL = f"https://finviz.com/quote.ashx?t={TICKER}"

## 1. Retrieve the Finviz snapshot

The scraper reads the label/value pairs rendered in Finviz's snapshot table. If Finviz blocks the request or changes its page structure, the notebook stops with a diagnostic message rather than silently returning incorrect data.

In [ ]:
async def scrape_finviz_snapshot(ticker: str) -> dict:
    url = f"https://finviz.com/quote.ashx?t={ticker.upper()}"
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/128.0.0.0 Safari/537.36"),
            viewport={"width": 1440, "height": 1000},
        )
        try:
            response = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_timeout(2500)
            title = (await page.title()).lower()
            body = (await page.locator("body").inner_text()).strip()
            if response is not None and response.status >= 400:
                raise RuntimeError(f"Finviz returned HTTP {response.status} for {url}")
            if any(term in title for term in ["access denied", "attention required", "just a moment"]):
                raise RuntimeError("Finviz returned an access/challenge page. This is a provider-side access restriction, not a parsing result.")
            if "snapshot" not in body.lower() and "forward p/e" not in body.lower():
                raise RuntimeError("Finviz page loaded, but the expected stock snapshot was not detected. The page may be blocked or its layout may have changed.")

            pairs = {}
            # Prefer the known snapshot table selectors, then inspect all tables as a fallback.
            tables = page.locator("table.snapshot-table2, table.js-snapshot-table, table")
            table_count = await tables.count()
            for table_index in range(table_count):
                cells = await tables.nth(table_index).locator("td").all_inner_texts()
                cleaned = [re.sub(r"\\s+", " ", value).strip() for value in cells]
                for i in range(0, len(cleaned) - 1, 2):
                    label, value = cleaned[i], cleaned[i + 1]
                    if label and value and label not in pairs:
                        pairs[label] = value
            if not pairs:
                raise RuntimeError("No label/value pairs were extracted from the Finviz page.")
            return pairs
        finally:
            await browser.close()

snapshot = await scrape_finviz_snapshot(TICKER)
print(f"Retrieved {len(snapshot)} Finviz snapshot fields for {TICKER}.")
display(pd.DataFrame(snapshot.items(), columns=["Finviz Field", "Value"]))

## 2. Select and validate the requested metrics

In [ ]:
def to_number(value):
    text = str(value).strip().replace(",", "")
    if text in {"-", "", "N/A", "nan"}:
        return float("nan")
    multiplier = 1
    if text.endswith("%"):
        text = text[:-1]
    if text.endswith("B"):
        multiplier, text = 1e9, text[:-1]
    elif text.endswith("M"):
        multiplier, text = 1e6, text[:-1]
    elif text.endswith("K"):
        multiplier, text = 1e3, text[:-1]
    try:
        return float(text) * multiplier
    except ValueError:
        return float("nan")

normalized = {re.sub(r"\\s+", " ", k).strip().lower(): v for k, v in snapshot.items()}
label_map = {
    "P/E": "p/e",
    "Forward P/E": "forward p/e",
    "P/S": "p/s",
    "P/C": "p/c",
    "P/B": "p/b",
    "P/FCF": "p/fcf",
}
missing = [label for label, key in label_map.items() if key not in normalized]
if missing:
    raise ValueError(f"Finviz did not provide these expected fields: {missing}. Available labels: {sorted(normalized)}")

metrics = pd.DataFrame([{
    "Ticker": TICKER.upper(),
    **{label: to_number(normalized[key]) for label, key in label_map.items()},
}])
display(metrics)
json_path = OUTPUT_DIR / f"{TICKER.upper()}_valuation_metrics.json"
metrics.to_json(json_path, orient="records", indent=2)
print(f"JSON saved to: {json_path}")

## 3. Visualize valuation multiples

Each multiple is shown separately because the ratios can have very different scales. Missing or non-numeric values are omitted from the chart but remain visible as missing values in the table.

In [ ]:
plot_data = metrics.set_index("Ticker").loc[TICKER.upper(), METRICS].dropna()
fig, ax = plt.subplots(figsize=(10, 5))
if plot_data.empty:
    ax.text(0.5, 0.5, "No numeric valuation multiples available from Finviz",
            ha="center", va="center", transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
else:
    plot_data.plot(kind="bar", ax=ax)
    ax.set_xlabel("Metric")
    ax.set_ylabel("Multiple (x)")
    ax.tick_params(axis="x", rotation=0)
ax.set_title(f"{TICKER.upper()} — Finviz valuation multiples")
fig.tight_layout()
chart_path = OUTPUT_DIR / f"{TICKER.upper()}_valuation_multiples.png"
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Chart saved to: {chart_path}")


## Notes on access and data quality

- Finviz remains the sole data provider in this project.
- A successful page load does not guarantee that every metric is populated; unavailable values are represented as `NaN`.
- HTTP errors, access challenges, and unexpected page layouts raise explicit errors.
- Automated browser access may be restricted by Finviz. The notebook does not attempt to bypass access controls; retry later or use an authorized access method if Finviz blocks the request.
- The output reflects the values visible on Finviz when the notebook runs.